# 17.15 平均誤差很小，答案為什麼仍會改變？


只有兩候選EOS與「貓」。原分數`[1,1.005]`選貓；量化後變`[1.01,1.005]`，改選EOS，即停止回答。平均絕對差僅`(0.01+0)/2=0.005`，卻跨過了第一名邊界。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

tokens = ["<EOS>", "貓"]
original = torch.tensor([1.0, 1.005])
quantized = torch.tensor([1.01, 1.005])
mae = (original - quantized).abs().mean().item()
print("分數MAE", round(mae, 4))
print("原版選擇", tokens[original.argmax().item()])
print("量化版選擇", tokens[quantized.argmax().item()])

`argmax`取最大分數索引，再查候選名稱。輸出0.005、貓、EOS。這是手設分數的單步反例，沒有證明真模型必定提早停；它說明平均數不能保證離散選擇不變。下一步前文改變後，差異還可能累積。

真正比較讓原版與量化版讀同樣提示，用同樣輸入處理和生成設定。既看標準答案的預測代價，也實際生成，分開核對內容、範圍、格式和正常EOS。若用隨機抽樣，兩版配對設相同種子，另換幾個種子觀察；最高分生成則記爲greedy。

已保存的weight-only PTQ實驗在十道最後屬性題上，FP32、4-bit、8-bit的生成ID完全相同，都答對6／10，但驗證五題的答對數爲2、1、2。這支持的是該批輸入的觀察，不能推成所有提示都相同，也沒有驗證JSON或安全情境。

品質檢查要與文件、記憶體和速度一起報告。17.14的QAT比較是另一份共同起點與更新預算，不能將不同更新步數的PTQ拿來當作只改變QAT的對照。

練習將兩版第二候選分數都改2。MAE仍0.005，卻都選貓。相同誤差在不同排名餘裕下，可以有不同後果。

<details>
<summary>補充：既有實驗、來源與完整測量範圍</summary>

這次真正量化實跑的最後10題，FP32／4-bit／8-bit都答對6題，完成匹配也都是6/10，EOS則都是10/10；三版的十筆原始生成ID逐項相同，沒有非法控制ID。可是共同69個回答與EOS目標的平均負對數代價（NLL，越低表示標準答案機率越高）分別為0.4565／0.4278／0.4554。代價能改變，最高分答案卻剛好沒跨過邊界，不能由這輪答案相同推論所有提示都相同。驗證五題更直接顯示差異：依FP32／4-bit／8-bit順序，答對2／1／2題，共同有效目標36個，代價約0.9151／0.9291／0.9118。量化沒有補上來源的能力缺口：`color=blue;shape=circle;pitch=low;color?`仍答`re`，並未答對`blue`。完整逐題結果保存在[量化實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/quantization.json)，本輪只測合成屬性問答，沒有測JSON、日期澄清或安全拒絕。

[17.14的QAT](https://birdhackor.github.io/tiny-perceptron-vlm/17.14.html)則保留了一個整模型反例：同樣350次更新後，普通微調完成後才量化固定權重（訓練後量化，post-training quantization，PTQ），答6/10；QAT在微調時已模擬量化，再打包則答4/10。後者將紅色方形的`describe`答成`circle`，前者答`square`；兩者都正常EOS結束。比較要沿同一個實驗內的共同底座與更新預算進行，不能把另一組120次更新的量化模型混進來，當成只改QAT的對照。

</details>
